# Pair Trading Dashboard

HDFC Bank (`HDFCBANK.NS`) and Kotak Mahindra Bank (`KOTAKBANK.NS`)


In [1]:
from datetime import date, timedelta

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import ipywidgets as widgets
from IPython.display import display, HTML

display(HTML("""
<style>
.dashboard-title { font-size: 24px; font-weight: 600; margin: 4px 0 2px; }
.dashboard-note { color: #667085; margin-bottom: 14px; }
.metric-row { display: flex; gap: 10px; margin: 12px 0 18px; }
.metric { border: 1px solid #e4e7ec; border-radius: 6px; padding: 10px 16px; min-width: 145px; background: #fff; }
.metric-label { color: #667085; font-size: 12px; }
.metric-value { color: #101828; font-size: 20px; font-weight: 600; margin-top: 3px; }
</style>
<div class='dashboard-title'>Pair Trading Dashboard</div>
<div class='dashboard-note'>HDFC Bank and Kotak Mahindra Bank · NSE daily prices</div>
"""))


In [2]:
today = date.today()
default_start = today - timedelta(days=365 * 3)
start_picker = widgets.DatePicker(description="From", value=default_start)
end_picker = widgets.DatePicker(description="To", value=today)
window_slider = widgets.IntSlider(description="Window", value=60, min=20, max=180, step=10, continuous_update=False)
threshold_slider = widgets.FloatSlider(description="Z threshold", value=2.0, min=1.0, max=3.5, step=0.1, readout_format=".1f", continuous_update=False)
refresh_button = widgets.Button(description="Refresh", button_style="primary", icon="refresh")
controls = widgets.HBox([start_picker, end_picker, window_slider, threshold_slider, refresh_button])
display(controls)


In [3]:
output = widgets.Output()
display(output)
def _format_pct(value):
    return "—" if pd.isna(value) else f"{value:.2f}%"

def _metric(label, value):
    return f"<div class='metric'><div class='metric-label'>{label}</div><div class='metric-value'>{value}</div></div>"

def build_dashboard(_=None):
    with output:
        output.clear_output(wait=True)
        if start_picker.value is None or end_picker.value is None or start_picker.value >= end_picker.value:
            display(HTML("<b>Choose a valid date range.</b>"))
            return
        try:
            prices = yf.download(["HDFCBANK.NS", "KOTAKBANK.NS"], start=start_picker.value, end=end_picker.value + timedelta(days=1), auto_adjust=True, progress=False)["Close"].dropna()
        except Exception as exc:
            display(HTML(f"<b>Data download failed:</b> {exc}"))
            return
        prices.columns = ["HDFC Bank", "Kotak Mahindra Bank"]
        if prices.empty:
            display(HTML("<b>No price data was returned for this range.</b>"))
            return
        log_prices = np.log(prices)
        hedge_ratio = np.polyfit(log_prices["Kotak Mahindra Bank"], log_prices["HDFC Bank"], 1)[0]
        spread = log_prices["HDFC Bank"] - hedge_ratio * log_prices["Kotak Mahindra Bank"]
        rolling_mean = spread.rolling(window_slider.value).mean()
        rolling_std = spread.rolling(window_slider.value).std()
        z_score = (spread - rolling_mean) / rolling_std
        signal = pd.Series(0, index=prices.index, dtype=float)
        signal[z_score > threshold_slider.value] = -1
        signal[z_score < -threshold_slider.value] = 1
        position = signal.replace(0, np.nan).ffill().fillna(0).shift(1).fillna(0)
        strategy_return = position * spread.diff()
        cumulative_strategy = strategy_return.cumsum()
        cumulative_spread = spread.diff().cumsum()
        entries = position.diff().fillna(position).loc[lambda x: x != 0]
        trades = pd.DataFrame({"Date": entries.index, "Position": entries.values})
        trades["Action"] = trades["Position"].map({1.0: "Long spread", -1.0: "Short spread", 0.0: "Flat"})
        trades = trades[["Date", "Action"]].sort_values("Date", ascending=False)
        total_return = (np.exp(cumulative_strategy.iloc[-1]) - 1) * 100
        spread_return_pct = (np.exp(cumulative_spread.iloc[-1]) - 1) * 100
        volatility = strategy_return.std() * np.sqrt(252) * 100
        sharpe = (strategy_return.mean() / strategy_return.std()) * np.sqrt(252) if strategy_return.std() else np.nan
        latest_z = z_score.iloc[-1]
        display(HTML("<div class='metric-row'>" + "".join([_metric("Strategy return", _format_pct(total_return)), _metric("Spread return", _format_pct(spread_return_pct)), _metric("Annualised volatility", _format_pct(volatility)), _metric("Sharpe ratio", "—" if pd.isna(sharpe) else f"{sharpe:.2f}"), _metric("Latest z-score", "—" if pd.isna(latest_z) else f"{latest_z:.2f}")]) + "</div>"))
        fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.08, subplot_titles=("Normalised prices", "Spread and trading bands", "Cumulative performance"))
        normalised = prices / prices.iloc[0] * 100
        for column in normalised.columns:
            fig.add_trace(go.Scatter(x=normalised.index, y=normalised[column], name=column, mode="lines"), row=1, col=1)
        fig.add_trace(go.Scatter(x=spread.index, y=spread, name="Spread", line=dict(color="#344054")), row=2, col=1)
        fig.add_trace(go.Scatter(x=spread.index, y=rolling_mean + threshold_slider.value * rolling_std, name="Upper band", line=dict(color="#98A2B3", dash="dot")), row=2, col=1)
        fig.add_trace(go.Scatter(x=spread.index, y=rolling_mean - threshold_slider.value * rolling_std, name="Lower band", line=dict(color="#98A2B3", dash="dot")), row=2, col=1)
        fig.add_trace(go.Scatter(x=cumulative_strategy.index, y=np.exp(cumulative_strategy) * 100, name="Strategy", line=dict(color="#1570EF")), row=3, col=1)
        fig.add_trace(go.Scatter(x=cumulative_spread.index, y=np.exp(cumulative_spread) * 100, name="Spread", line=dict(color="#98A2B3")), row=3, col=1)
        long_entries = trades[trades["Action"] == "Long spread"]["Date"]
        short_entries = trades[trades["Action"] == "Short spread"]["Date"]
        fig.add_trace(go.Scatter(x=long_entries, y=spread.reindex(long_entries), mode="markers", name="Long entry", marker=dict(color="#12B76A", size=8, symbol="triangle-up")), row=2, col=1)
        fig.add_trace(go.Scatter(x=short_entries, y=spread.reindex(short_entries), mode="markers", name="Short entry", marker=dict(color="#F04438", size=8, symbol="triangle-down")), row=2, col=1)
        fig.update_layout(height=850, template="plotly_white", hovermode="x unified", margin=dict(l=55, r=25, t=55, b=35), legend=dict(orientation="h", y=1.03, x=0))
        fig.update_yaxes(showgrid=True, gridcolor="#F2F4F7")
        display(fig)
        display(HTML("<h4>Recent signal changes</h4>"))
        display(trades.head(12).style.format({"Date": lambda value: value.strftime("%Y-%m-%d")}))

refresh_button.on_click(build_dashboard)
build_dashboard()


Output()